# 02 — Data Quality and Exploratory Data Analysis

This notebook explores historical Instacart order behavior using the optimized silver-layer tables created in Notebook 01. The goal is to identify reorder patterns, shopping timing behavior, product reliability issues, and user-level signals that motivate the downstream reorder prediction model.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
base_path = "/Volumes/workspace/default/instacart"  # change if yours is different
silver_path = f"{base_path}/silver"
eda_path = f"{base_path}/eda_outputs"

dbutils.fs.mkdirs(eda_path)

In [0]:
prior = spark.read.parquet(f"{silver_path}/prior_order_details")
train = spark.read.parquet(f"{silver_path}/train_order_details")
products = spark.read.parquet(f"{silver_path}/product_details")

In [0]:
df = prior

For EDA, only historical `prior` orders are used. The `train` order set is reserved for supervised modeling labels in later notebooks, so keeping EDA focused on prior behavior helps preserve a clean modeling flow.

## 1. Basic Data Quality Checks

In [0]:
print("Prior rows:", prior.count())
print("Train rows:", train.count())
print("Products:", products.count())

In [0]:
df.printSchema()

In [0]:
missing_summary = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

display(missing_summary)

`days_since_prior_order` contains null values because a user's first order has no previous order. These nulls are expected and are not treated as data quality errors. Product metadata nulls were already handled in the ingestion notebook by assigning unknown aisle/department values.

## 2. EDA Questions Answered

### Question 1: What are the most purchased products?

In [0]:
top_products = (
    df.groupBy("product_id", "product_name", "aisle", "department")
      .agg(
          F.count("*").alias("total_purchases"),
          F.sum("reordered").alias("total_reorders"),
          F.avg("reordered").alias("raw_reorder_rate")
      )
      .orderBy(F.desc("total_purchases"))
)

display(top_products.limit(20))

In [0]:
top_products.write.mode("overwrite").parquet(f"{eda_path}/top_products")

### Question 2: Which products have the highest reorder rate?

Raw reorder rate can be misleading for low-volume products. For example, a product purchased only a few times can show a 100% reorder rate by chance. To make product rankings more reliable, reorder behavior is evaluated using both a minimum purchase threshold and a smoothed reorder rate.

Applying minimum support threshold

In [0]:
reliable_reorder_products = (
    top_products
    .filter(F.col("total_purchases") >= 500)
    .orderBy(F.desc("raw_reorder_rate"))
)

display(reliable_reorder_products.limit(20))

Added smoothed reorder rate

In [0]:
global_reorder_rate = df.agg(F.avg("reordered")).first()[0]
global_reorder_rate

In [0]:
smoothing_factor = 100

product_reorder_smoothed = (
    top_products
    .withColumn(
        "smoothed_reorder_rate",
        (F.col("total_reorders") + F.lit(global_reorder_rate * smoothing_factor)) /
        (F.col("total_purchases") + F.lit(smoothing_factor))
    )
    .orderBy(F.desc("smoothed_reorder_rate"))
)

display(product_reorder_smoothed.limit(20))

In [0]:
product_reorder_smoothed.write.mode("overwrite").parquet(f"{eda_path}/product_reorder_smoothed")

### Question 3: Which departments drive reorder behavior?

In [0]:
department_summary = (
    df.groupBy("department")
      .agg(
          F.count("*").alias("total_purchases"),
          F.sum("reordered").alias("total_reorders"),
          F.avg("reordered").alias("reorder_rate"),
          F.countDistinct("product_id").alias("unique_products"),
          F.countDistinct("order_id").alias("unique_orders")
      )
      .orderBy(F.desc("total_purchases"))
)

display(department_summary)

In [0]:
department_summary.write.mode("overwrite").parquet(f"{eda_path}/department_summary")

### Question 4: When do people order?

#### Orders by day of week

In [0]:
orders_only = (
    df.select("order_id", "user_id", "order_dow", "order_hour_of_day", "days_since_prior_order")
      .dropDuplicates(["order_id"])
)

In [0]:
orders_by_dow = (
    orders_only.groupBy("order_dow")
      .agg(F.count("*").alias("num_orders"))
      .orderBy("order_dow")
)

display(orders_by_dow)

#### Orders by hour

In [0]:
orders_by_hour = (
    orders_only.groupBy("order_hour_of_day")
      .agg(F.count("*").alias("num_orders"))
      .orderBy("order_hour_of_day")
)

display(orders_by_hour)

#### Day-hour heatmap table

In [0]:
dow_hour = (
    orders_only.groupBy("order_dow", "order_hour_of_day")
      .agg(F.count("*").alias("num_orders"))
      .orderBy("order_dow", "order_hour_of_day")
)

display(dow_hour)

In [0]:
orders_by_dow.write.mode("overwrite").parquet(f"{eda_path}/orders_by_dow")
orders_by_hour.write.mode("overwrite").parquet(f"{eda_path}/orders_by_hour")
dow_hour.write.mode("overwrite").parquet(f"{eda_path}/dow_hour_orders")

### Question 5: How often do customers reorder?

In [0]:
days_between_orders = (
    orders_only
    .filter(F.col("days_since_prior_order").isNotNull())
    .groupBy("days_since_prior_order")
    .agg(F.count("*").alias("num_orders"))
    .orderBy("days_since_prior_order")
)

display(days_between_orders)

In [0]:
display(
    orders_only
    .select("days_since_prior_order")
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)

### Question 6: Which products are added early to carts?

In [0]:
cart_position_summary = (
    df.groupBy("product_id", "product_name", "department", "aisle")
      .agg(
          F.count("*").alias("total_purchases"),
          F.avg("add_to_cart_order").alias("avg_cart_position"),
          F.expr("percentile_approx(add_to_cart_order, 0.5)").alias("median_cart_position"),
          F.avg("reordered").alias("reorder_rate")
      )
      .filter(F.col("total_purchases") >= 500)
      .orderBy("avg_cart_position")
)

display(cart_position_summary.limit(20))

In [0]:
display(
    cart_position_summary
    .orderBy(F.desc("avg_cart_position"))
    .limit(20)
)

In [0]:
cart_position_summary.write.mode("overwrite").parquet(f"{eda_path}/cart_position_summary")

Early-cart products are likely planned staples; late-cart products may be impulse or complementary items.

### Question 7: Which aisles/departments have large baskets?

In [0]:
order_basket_size = (
    df.groupBy("order_id")
      .agg(F.count("*").alias("basket_size"))
)

In [0]:
display(
    order_basket_size
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)

In [0]:
order_dept_counts = (
    df.groupBy("order_id", "department")
      .agg(F.count("*").alias("dept_items"))
)

dept_basket_summary = (
    order_dept_counts.groupBy("department")
      .agg(
          F.avg("dept_items").alias("avg_items_per_order_when_present"),
          F.countDistinct("order_id").alias("orders_containing_department")
      )
      .orderBy(F.desc("orders_containing_department"))
)

display(dept_basket_summary)

In [0]:
order_basket_size.write.mode("overwrite").parquet(f"{eda_path}/order_basket_size")
dept_basket_summary.write.mode("overwrite").parquet(f"{eda_path}/dept_basket_summary")

### Question 8: User-level shopping behavior

In [0]:
user_summary = (
    df.groupBy("user_id")
      .agg(
          F.countDistinct("order_id").alias("total_orders"),
          F.count("*").alias("total_items"),
          F.countDistinct("product_id").alias("unique_products"),
          F.avg("reordered").alias("user_reorder_rate"),
          F.avg("add_to_cart_order").alias("avg_cart_position"),
          F.avg("days_since_prior_order").alias("avg_days_between_orders")
      )
      .withColumn("avg_basket_size", F.col("total_items") / F.col("total_orders"))
)

display(user_summary.limit(20))

In [0]:
display(
    user_summary
    .select(
        "total_orders",
        "total_items",
        "unique_products",
        "user_reorder_rate",
        "avg_days_between_orders",
        "avg_basket_size"
    )
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)

In [0]:
user_summary.write.mode("overwrite").parquet(f"{eda_path}/user_summary")

### Question 9: Are reorders different by hour/day?

In [0]:
reorder_by_hour = (
    df.groupBy("order_hour_of_day")
      .agg(
          F.count("*").alias("total_items"),
          F.avg("reordered").alias("reorder_rate")
      )
      .orderBy("order_hour_of_day")
)

display(reorder_by_hour)

In [0]:
reorder_by_dow = (
    df.groupBy("order_dow")
      .agg(
          F.count("*").alias("total_items"),
          F.avg("reordered").alias("reorder_rate")
      )
      .orderBy("order_dow")
)

display(reorder_by_dow)

In [0]:
reorder_by_hour.write.mode("overwrite").parquet(f"{eda_path}/reorder_by_hour")
reorder_by_dow.write.mode("overwrite").parquet(f"{eda_path}/reorder_by_dow")

This tells you whether certain shopping times are more routine/replenishment-driven.

### Question 10: Department affinity by user

In [0]:
user_department_counts = (
    df.groupBy("user_id", "department")
      .agg(F.count("*").alias("dept_purchase_count"))
)

In [0]:
w = Window.partitionBy("user_id").orderBy(F.desc("dept_purchase_count"))

user_top_department = (
    user_department_counts
    .withColumn("rank", F.row_number().over(w))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

display(user_top_department.limit(20))

In [0]:
top_department_distribution = (
    user_top_department.groupBy("department")
      .agg(F.count("*").alias("num_users"))
      .orderBy(F.desc("num_users"))
)

display(top_department_distribution)

In [0]:
user_top_department.write.mode("overwrite").parquet(f"{eda_path}/user_top_department")
top_department_distribution.write.mode("overwrite").parquet(f"{eda_path}/top_department_distribution")

#### New vs reordered product behavior

In [0]:
new_vs_reordered = (
    df.groupBy("department", "reordered")
      .agg(F.count("*").alias("item_count"))
      .orderBy("department", "reordered")
)

display(new_vs_reordered)

In [0]:
department_new_reorder = (
    df.groupBy("department")
      .agg(
          F.sum(F.when(F.col("reordered") == 1, 1).otherwise(0)).alias("reordered_items"),
          F.sum(F.when(F.col("reordered") == 0, 1).otherwise(0)).alias("new_items"),
          F.count("*").alias("total_items")
      )
      .withColumn("reorder_rate", F.col("reordered_items") / F.col("total_items"))
      .withColumn("new_item_rate", F.col("new_items") / F.col("total_items"))
      .orderBy(F.desc("reorder_rate"))
)

display(department_new_reorder)

In [0]:
department_new_reorder.write.mode("overwrite").parquet(
    f"{eda_path}/department_new_reorder"
)

In [0]:
summary_stats = {
    "num_prior_rows": prior.count(),
    "num_train_rows": train.count(),
    "num_users": df.select("user_id").distinct().count(),
    "num_orders": df.select("order_id").distinct().count(),
    "num_products": df.select("product_id").distinct().count(),
    "num_departments": df.select("department").distinct().count(),
    "global_reorder_rate": global_reorder_rate
}

summary_stats

In [0]:
summary_stats_df = spark.createDataFrame(
    [(k, float(v)) for k, v in summary_stats.items()],
    ["metric", "value"]
)

display(summary_stats_df)

summary_stats_df.write.mode("overwrite").parquet(f"{eda_path}/summary_stats")

## EDA Summary

The exploratory analysis shows that Instacart purchasing behavior is highly repeat-driven, with reorder patterns varying across products, departments, users, and shopping time. Simple product popularity is not enough for recommendation because high-volume products and high-reorder products capture different business signals.

To avoid misleading conclusions, product reorder rankings were evaluated using minimum purchase thresholds and smoothed reorder rates. This prevents low-volume products from dominating rankings due to artificially high reorder percentages.

Key modeling implications:

- User-product history is likely the strongest signal for reorder prediction.
- Product-level reorder rate captures global product loyalty.
- User-level reorder rate captures customer routine behavior.
- Days since prior order and order timing may help identify replenishment cycles.
- Cart position can distinguish staple products from impulse or add-on products.
- Department and aisle features can support both modeling and business segmentation.
- User-product interaction history, because repeat purchases are central to grocery behavior.
- Product-level reorder behavior, but with smoothing to avoid overvaluing low-volume products.
- User-level reorder tendency, since customers vary in how routine or exploratory their shopping patterns are.
- Recency and timing features, because grocery purchases often follow weekly or replenishment cycles.
- Cart position, because early-cart items may represent planned staples while late-cart items may represent impulse or complementary purchases.


These findings directly motivate the next stage: leakage-safe feature engineering for next-basket reorder prediction.